In [ ]:
!pip install imbalanced-learn xgboost lightgbm seaborn matplotlib pandas scikit-learn

  Using cached imbalanced_learn-0.12.4-py3-none-any.whl.metadata (8.3 kB)
  Using cached xgboost-1.6.2-py3-none-win_amd64.whl.metadata (1.8 kB)
  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
  Using cached numpy-1.21.6-cp37-cp37m-win_amd64.whl.metadata (2.2 kB)
  Using cached scipy-1.7.3-cp37-cp37m-win_amd64.whl.metadata (2.2 kB)
  Using cached scikit_learn-1.0.2-cp37-cp37m-win_amd64.whl.metadata (10 kB)
  Using cached joblib-1.3.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached threadpoolctl-3.1.0-py3-none-any.whl.metadata (9.2 kB)
Using cached imbalanced_learn-0.12.4-py3-none-any.whl (258 kB)
Using cached xgboost-1.6.2-py3-none-win_amd64.whl (125.4 MB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)
   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.1 MB ? eta -:--:--
   ---------------------------------------- 0.1/7.1 MB 991.0 kB/s eta 0:00:08
    --------------------------------

DEPRECATION: pandas 0.23.4 has a non-standard dependency specifier pytz>=2011k. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of pandas or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

ModuleNotFoundError: No module named 'imblearn'

In [ ]:
import re
data_path = 'creditcard_processed.csv'
df = pd.read_csv(data_path)

# Xóa cột Unnamed: 0 nếu tồn tại và loại bỏ ký tự đặc biệt trong tên cột cho LightGBM
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)
df = df.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))

print(f"Dataset shape: {df.shape}")
display(df.head())

In [ ]:
print("Class Distribution:")
print(df['Class'].value_counts())
print("\nPercentage of Fraudulent Transactions:", 
      round(df['Class'].value_counts()[1] / len(df) * 100, 3), "%")

plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=df)
plt.title('Class Distribution')
plt.show()

In [ ]:
# Tách feature và target
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

In [ ]:
# 1. Original (Mặc định)
# 2. SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# 3. Random UnderSampling
rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

# 4. Random OverSampling
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

print("Original label distribution:", np.bincount(y_train))
print("SMOTE label distribution:", np.bincount(y_train_smote))
print("UnderSampling label distribution:", np.bincount(y_train_rus))
print("OverSampling label distribution:", np.bincount(y_train_ros))

datasets = {
    'Original': (X_train, y_train),
    'SMOTE': (X_train_smote, y_train_smote),
    'UnderSampling': (X_train_rus, y_train_rus),
    'OverSampling': (X_train_ros, y_train_ros)
}

Khởi tạo Hàm Đánh giá

In [ ]:
def evaluate_model(y_true, y_pred, model_name, prep_method):
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    
    print(f"--- {model_name} with {prep_method} ---")
    print(f"Accuracy : {acc:.5f}")
    print(f"Precision: {prec:.5f}")
    print(f"Recall   : {rec:.5f}")
    print(f"F1-Score : {f1:.5f}")
    print(classification_report(y_true, y_pred))
    
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix\n{model_name} ({prep_method})')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()
    
    return {'Model': model_name, 'Method': prep_method, 'F1-Score': f1, 'Accuracy': acc, 'Precision': prec, 'Recall': rec}

global_results = []

Huấn luyện Mô hình: Random Forest

In [ ]:
print("=========== RANDOM FOREST ===========")
m_name = 'Random Forest'
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)

for d_name, (X_tr, y_tr) in datasets.items():
    start_time = time.time()
    rf_model.fit(X_tr, y_tr)
    y_pred = rf_model.predict(X_test)
    
    res = evaluate_model(y_test, y_pred, m_name, d_name)
    res['Training Time (s)'] = round(time.time() - start_time, 2)
    global_results.append(res)
    print("-" * 50)

Huấn luyện Mô hình: XGBoost

In [ ]:
print("=========== XGBOOST ===========")
m_name = 'XGBoost'
xgb_model = xgb.XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')

for d_name, (X_tr, y_tr) in datasets.items():
    start_time = time.time()
    xgb_model.fit(X_tr, y_tr)
    y_pred = xgb_model.predict(X_test)
    
    res = evaluate_model(y_test, y_pred, m_name, d_name)
    res['Training Time (s)'] = round(time.time() - start_time, 2)
    global_results.append(res)
    print("-" * 50)

Huấn luyện Mô hình: LightGBM

In [ ]:
print("=========== LIGHTGBM ===========")
m_name = 'LightGBM'
lgb_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

for d_name, (X_tr, y_tr) in datasets.items():
    start_time = time.time()
    lgb_model.fit(X_tr, y_tr)
    y_pred = lgb_model.predict(X_test)
    
    res = evaluate_model(y_test, y_pred, m_name, d_name)
    res['Training Time (s)'] = round(time.time() - start_time, 2)
    global_results.append(res)
    print("-" * 50)

Tổng hợp Kết quả đánh giá

In [ ]:
results_df = pd.DataFrame(global_results)
display(results_df.sort_values(by='F1-Score', ascending=False))

In [ ]:
plt.figure(figsize=(15, 6))
sns.barplot(data=results_df, x='Model', y='F1-Score', hue='Method')
plt.title('F1-Score Comparison Across Models and Sampling Methods')
plt.ylim(0, 1.0)
plt.show()

plt.figure(figsize=(15, 6))
sns.barplot(data=results_df, x='Model', y='Precision', hue='Method')
plt.title('Precision Comparison Across Models and Sampling Methods')
plt.ylim(0, 1.0)
plt.show()

plt.figure(figsize=(15, 6))
sns.barplot(data=results_df, x='Model', y='Recall', hue='Method')
plt.title('Recall Comparison Across Models and Sampling Methods')
plt.ylim(0, 1.0)
plt.show()